---

# Summary of fem2d Features

## What You've Seen

This notebook has demonstrated the key capabilities of **fem2d**:

### ✅ Linear Analysis
- **Frame structures** with rigid joints (Example 1)
- **Truss structures** with pin joints (Example 2)
- **Continuous beams** with distributed loads (Example 3)

### ✅ Non-linear Analysis
- **Geometrically non-linear beams** with large deflections (Example 4)
- **Geometrically non-linear trusses** with snap-through behavior (Example 5)

### ✅ Core Features
- **SimpleFrame**: High-level API for quick structure setup
- **Structure**: Low-level API for advanced customization
- **Results**: Post-processing of displacements, reactions, member forces
- **DrawStructure**: Visualization of geometry, loads, and deformed shapes
- **Newton-Raphson Solver**: Non-linear analysis with convergence control

## Getting Started with fem2d

### Installation
```bash
pip install fem2d
```

### Basic Workflow
1. Create a structure using `SimpleFrame`
2. Add nodes, elements, supports, and loads
3. Call `frame.solve()` for linear analysis or `structure.solve_nonlinear()` for non-linear
4. Extract results using the `Results` class
5. Visualize using `DrawStructure`

### Key Classes & Methods
- `SimpleFrame()`: High-level frame creation interface
- `SimpleFrame.add_node(id, x, y)`: Add a node
- `SimpleFrame.add_frame(id, node_i, node_j, E, A, I)`: Add beam element
- `SimpleFrame.add_truss(id, node_i, node_j, E, A)`: Add truss element
- `SimpleFrame.add_support(node_id, [ux, uy, rz])`: Apply boundary conditions
- `SimpleFrame.add_node_load(node_id, [fx, fy, mz])`: Apply point loads
- `SimpleFrame.add_distributed_load(elem_id, wy)`: Apply distributed loads
- `frame.solve()`: Linear analysis
- `Results(frame)`: Extract results
- `DrawStructure(structure)`: Visualize

## Next Steps

Try modifying one of these examples:
1. Change material properties
2. Add or modify loads
3. Add/remove supports
4. Change element properties
5. Run the non-linear solver instead of linear

For detailed API documentation, visit the fem2d docs or see the docstrings of the classes.

---

**Questions or Issues?** See the [fem2d repository](https://github.com/abinashmandal/fem-2d) for more information.

In [ ]:
# Visualize the beam structure from Example 3
if HAVE_DRAWING:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # Plot 1: Original undeformed structure
    plt.sca(ax1)
    drawer = DrawStructure(frame_beam.structure, scale=1.0, arrow_scale=0.05)
    drawer.draw()
    ax1.set_title("Example 3: Continuous Beam - Original Configuration", fontsize=12, fontweight='bold')
    ax1.set_aspect('equal')
    ax1.grid(True, alpha=0.3)
    
    # Plot 2: Deformed structure (magnified)
    plt.sca(ax2)
    drawer_deformed = DrawStructure(frame_beam.structure, scale=500.0, arrow_scale=0.05)
    drawer_deformed.draw()
    ax2.set_title("Example 3: Continuous Beam - Deformed Shape (500x magnified)", 
                  fontsize=12, fontweight='bold')
    ax2.set_aspect('equal')
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print("✓ Visualization complete")
else:
    print("Note: Install matplotlib to enable structure visualization")

---

# Visualization Examples

## Drawing Structures with fem2d

The `DrawStructure` utility allows you to visualize:
- Structure geometry
- Support conditions
- Applied loads
- Deformed shapes (magnified for visibility)

In [ ]:
# Example 10.2: Non-linear symmetric truss with snap-through behavior
# From Kassimali A. (2022), Matrix analysis of structures
# units in kN, m

# 1. Create nodes
node1_nl = Node(1, 0.0, 0.0)
node2_nl = Node(2, 4.0, 3.0)
node3_nl = Node(3, 8.0, 0.0)

# 2. Material properties
E_truss = 200e6  # 200 GPa = 200×10⁶ kN/m²
EA_truss = 45155.0  # EA stiffness
A_truss = EA_truss / E_truss

material_truss = ElasticMaterial(E_truss)

# 3. Create non-linear truss elements
elem1_nl = TrussElementNL(1, node1_nl, node2_nl, material_truss, A_truss)
elem2_nl = TrussElementNL(2, node2_nl, node3_nl, material_truss, A_truss)
elem3_nl = TrussElementNL(3, node3_nl, node1_nl, material_truss, A_truss)

# 4. Build structure
structure_truss_nl = Structure()
structure_truss_nl.add_node(node1_nl)
structure_truss_nl.add_node(node2_nl)
structure_truss_nl.add_node(node3_nl)
structure_truss_nl.add_element(elem1_nl)
structure_truss_nl.add_element(elem2_nl)
structure_truss_nl.add_element(elem3_nl)

# 5. Apply supports
node1_nl.set_support(ux_fixed=True, uy_fixed=True)  # Pinned support
node3_nl.set_support(ux_fixed=False, uy_fixed=True)  # Roller support

# 6. Apply external load (vertical load at node 2)
node2_nl.set_load(fx=0.0, fy=-2000.0, mz=0.0)

# 7. Run non-linear analysis
structure_truss_nl.solve_nonlinear(tolerance=1e-8, max_iter=30)

# Extract results
results_truss_nl = Results(structure_truss_nl)
disp_truss_nl = results_truss_nl.node_displacements()
reactions_truss_nl = results_truss_nl.reactions()
el_forces_truss_nl = results_truss_nl.element_forces()

print("=" * 60)
print("EXAMPLE 5: Non-linear Truss Analysis (Snap-through)")
print("=" * 60)
print(f"\nGeometry:")
print(f"  Node 1: (0.0, 0.0) - Pinned support")
print(f"  Node 2: (4.0, 3.0) - Load applied here")
print(f"  Node 3: (8.0, 0.0) - Roller support")
print(f"\nLoad: 2000 kN vertical downward")

print("\nNode Displacements:")
print(disp_truss_nl)
print("\nReactions:")
print(reactions_truss_nl)
print("\nElement Forces:")
print(el_forces_truss_nl)

print("\n" + "-" * 60)
print("Verification Results:")
print("-" * 60)
print(f"Node 2 vertical displacement: {disp_truss_nl['uy'][1]:.6f} m")
print(f"Element 3 axial force:        {el_forces_truss_nl['fx_i'][2]:.2f} kN")

## Example 5: Non-linear Truss Analysis (Snap-through)

This example shows a symmetric truss structure undergoing large deformations under a central vertical load.

**Reference:** Typical non-linear truss snap-through analysis  
**Units:** kN, m

In [ ]:
# Example 7.1.1 – Cantilever beam with end vertical load (non-linear)
# Demonstrates large deformation analysis using Newton-Raphson solver

# Geometry
L = 10.0  # Length in meters
b = 0.5   # Width in meters
h = 0.25  # Height in meters
A = b * h  # Cross-sectional area
I = b * h**3 / 12  # Moment of inertia
E = 200e9  # Young's modulus in Pa

material = ElasticMaterial(E)

# Create 5 elements (6 nodes) for discretization
n_elem = 5
x = np.linspace(0, L, n_elem + 1)
nodes_nl = []
for i, xi in enumerate(x):
    nodes_nl.append(Node(i + 1, xi, 0.0))

structure_nl = Structure()
for node in nodes_nl:
    structure_nl.add_node(node)

# Create non-linear beam elements
for i in range(n_elem):
    elem = BeamElementNL(i + 1, nodes_nl[i], nodes_nl[i + 1], material, A, I)
    structure_nl.add_element(elem)

# Support: cantilever fixed at node 1
nodes_nl[0].set_support(ux_fixed=True, uy_fixed=True, rz_fixed=True)

# Load: vertical load at tip
# Using dimensionless load parameter: PL²/EI = 10
EI = E * I
P_target = 10.0 * EI / L**2
nodes_nl[-1].set_load(fx=0.0, fy=-P_target, mz=0.0)

# Solve using non-linear solver (Newton-Raphson)
structure_nl.solve_nonlinear(tolerance=1e-8, max_iter=30)

print("=" * 60)
print("EXAMPLE 4: Non-linear Cantilever Beam (Large Deflections)")
print("=" * 60)
print(f"\nGeometry:")
print(f"  Length: {L} m")
print(f"  Width: {b} m")
print(f"  Height: {h} m")
print(f"\nMaterial:")
print(f"  Young's Modulus: {E/1e9} GPa")
print(f"  Load parameter (PL²/EI): 10.0")

# Extract tip displacements
disp_tip = structure_nl.disp[nodes_nl[-1].dofs]
print(f"\nTip Displacements:")
print(f"  ux = {disp_tip[0]:.6f} m   (U/L = {disp_tip[0]/L:.6f})")
print(f"  uy = {disp_tip[1]:.6f} m   (W/L = {disp_tip[1]/L:.6f})")
print(f"  theta = {disp_tip[2]:.6f} rad")

print(f"\nNote: Large deformation analysis accounts for geometric non-linearity")

---

# Non-linear Analysis Examples

## Example 4: Non-linear Cantilever Beam (Large Deflections)

This example demonstrates geometrically non-linear analysis of a cantilever beam with large end deflections.

**Reference:** Typical non-linear beam analysis  
**Units:** m, Pa

In [ ]:
# Example 5.6: Kassimali A. (2022), Matrix analysis of structures
# units in kN, m

frame_beam = SimpleFrame()

# Define nodes in a horizontal line
frame_beam.add_node(1, 0, 0)
frame_beam.add_node(2, 4, 0)
frame_beam.add_node(3, 7, 0)
frame_beam.add_node(4, 11, 0)

# Material and section properties
E = 200e6  # Young's modulus in kPa
I = 108e-6  # Moment of inertia in m^4
A = 10  # Cross-sectional area in cm^2

# Add frame elements (horizontal beam)
frame_beam.add_frame(1, 1, 2, E=E, A=A, I=I)
frame_beam.add_frame(2, 2, 3, E=E, A=A, I=I)
frame_beam.add_frame(3, 3, 4, E=E, A=A, I=I)

# Add supports
frame_beam.add_support(1, [1, 1, 1])  # Fixed support at left end
frame_beam.add_support(2, [0, 1, 0])  # Vertical support (pin) at node 2
frame_beam.add_support(3, [0, 1, 0])  # Vertical support (pin) at node 3
frame_beam.add_support(4, [1, 1, 1])  # Fixed support at right end

# Add loads
frame_beam.add_element_point_load(1, py=-150, x=2)  # Point load on element 1 at x=2m
frame_beam.add_distributed_load(3, wy=-37.5)  # Distributed load on element 3

# Solve
frame_beam.solve()

# Extract results
results_beam = Results(frame_beam)
disp_beam = results_beam.node_displacements()
reactions_beam = results_beam.reactions()
el_forces_beam = results_beam.element_forces()

print("=" * 60)
print("EXAMPLE 3: Continuous Beam (Kassimali 5.6)")
print("=" * 60)
print("\nNode Displacements:")
print(disp_beam)
print("\nReactions:")
print(reactions_beam)
print("\nElement Forces:")
print(el_forces_beam)

print("\n" + "-" * 60)
print("Verification Results:")
print("-" * 60)
print(f"Node 2 rotation (theta):     {disp_beam['theta'][1]:.10f} rad")
print(f"Element 2 shear at i-end:    {el_forces_beam['fy_i'][1]:.6f} kN")

## Example 3: Continuous Beam with Distributed Load (Kassimali 5.6)

This example demonstrates a continuous beam with multiple supports and a distributed load.

**Reference:** Kassimali, A. (2022). Matrix analysis of structures  
**Units:** kN, m

In [ ]:
# Problem: Logan D. L. (2017), A First Course in the Finite Element Method
# Triangular truss with 4 members
# units in kips, feet

frame_truss = SimpleFrame()
ft = 12  # Convert feet to inches

# Define nodes
frame_truss.add_node(1, 0, 0)
frame_truss.add_node(2, 60 * ft, 0)
frame_truss.add_node(3, 30 * ft, 40 * ft)
frame_truss.add_node(4, 30 * ft, 60 * ft)

# Material properties
E = 30e3  # Young's modulus in ksi
A = 3  # Cross-sectional area in sq in

# Add truss elements (only axial stiffness, no bending)
frame_truss.add_truss(1, 1, 3, E, A)
frame_truss.add_truss(2, 2, 3, E, A)
frame_truss.add_truss(3, 3, 4, E, A)

# Add supports (pinned supports at base nodes)
frame_truss.add_support(1, [1, 1, 1])
frame_truss.add_support(2, [1, 1, 1])
frame_truss.add_support(4, [1, 1, 1])

# Add load at node 3
frame_truss.add_node_load(3, [5, -10, 0])  # 5 kips horizontal, 10 kips vertical down

# Solve
frame_truss.solve()

# Extract results
results_truss = Results(frame_truss.structure)
disp_truss = results_truss.node_displacements()
reactions_truss = results_truss.reactions()
el_forces_truss = results_truss.element_forces()

print("=" * 60)
print("EXAMPLE 2: Truss Analysis (Logan Truss)")
print("=" * 60)
print("\nNode Displacements:")
print(disp_truss)
print("\nReactions:")
print(reactions_truss)
print("\nElement Forces:")
print(el_forces_truss)

print("\n" + "-" * 60)
print("Verification Results:")
print("-" * 60)
print(f"Node 3 vertical displacement: {disp_truss['uy'][2]:.10f} in")
print(f"Element 2 axial force:        {el_forces_truss['fx_i'][1]:.6f} kips")

## Example 2: Truss Analysis (Logan Truss)

This example analyzes a triangular truss structure with fixed supports and a vertical point load.

**Reference:** Logan, D. L. (2017). A First Course in the Finite Element Method  
**Units:** kips, feet

In [ ]:
# Example 5.1: Logan D. L. (2017), A First Course in the Finite Element Method
# units in kips, inches

frame = SimpleFrame()

# Define nodes
frame.add_node(1, 0, 0)
frame.add_node(2, 0, 10 * 12)  # 10 feet
frame.add_node(3, 10 * 12, 10 * 12)  # 10 feet horizontal
frame.add_node(4, 10 * 12, 0)

# Material and section properties
E = 30e3  # Young's modulus in ksi
A = 10  # Cross-sectional area in sq in
I = 200  # Moment of inertia in in^4

# Add frame elements (beams)
frame.add_frame(1, 1, 2, E, A, I)
frame.add_frame(2, 2, 3, E, A, I / 2)
frame.add_frame(3, 3, 4, E, A, I)

# Add supports (fixed at nodes 1 and 4)
frame.add_support(1, [1, 1, 1])  # Fixed support (ux, uy, rz all constrained)
frame.add_support(4, [1, 1, 1])  # Fixed support

# Add loads
frame.add_node_load(2, [10, 0, 0])  # 10 kips horizontal at node 2
frame.add_node_load(3, [0, 0, 5])   # 5 kip-inches moment at node 3

# Solve the system
frame.solve()

# Extract results
results = Results(frame)
disp = results.node_displacements()
reactions = results.reactions()
el_forces = results.element_forces()

print("=" * 60)
print("EXAMPLE 1: Frame Analysis (Logan 5.1)")
print("=" * 60)
print("\nNode Displacements:")
print(disp)
print("\nReactions:")
print(reactions)
print("\nElement Forces:")
print(el_forces)

# Verify specific results
print("\n" + "-" * 60)
print("Verification Results:")
print("-" * 60)
print(f"Node 2 rotation (theta):     {disp['theta'][1]:.10f}")
print(f"Element 2 moment at i-end:   {el_forces['m_i'][1]:.6f} kip-in")

---

# Linear Analysis Examples

## Example 1: Frame Analysis (Logan 5.1)

This example analyzes a rigid frame with fixed supports at nodes 1 and 4, and point loads at nodes 2 and 3.

**Reference:** Logan, D. L. (2017). A First Course in the Finite Element Method  
**Units:** kips, inches

In [ ]:
# Import main classes from fem2d
from fem2d import SimpleFrame, Structure, Node, ElasticMaterial
from fem2d import BeamElement, BeamElementNL, TrussElement, TrussElementNL
from fem2d import NewtonRaphsonSolver
from fem2d.results import Results

try:
    from fem2d.utils import DrawStructure
    HAVE_DRAWING = True
except ImportError:
    HAVE_DRAWING = False
    print("Note: DrawStructure requires matplotlib")

print("✓ fem2d library imported successfully")
print(f"  - SimpleFrame: {SimpleFrame}")
print(f"  - DrawStructure available: {HAVE_DRAWING}")

## Import fem2d Library

In [ ]:
import sys
import os
from pathlib import Path
import numpy as np
import pandas as pd

# Add project root to path
project_root = Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Configure matplotlib for inline display
%matplotlib inline
import matplotlib.pyplot as plt

print(f"Working directory: {project_root}")
print(f"Python version: {sys.version}")

---

## Setup and Imports

# fem2d Examples Showcase

Welcome to the **fem2d** library examples notebook! 

This notebook demonstrates how to use the fem2d library for 2D finite element analysis of structural frames, beams, and trusses. Included are both **linear** and **geometrically non-linear** analysis examples.

## What is fem2d?

**fem2d** is a Python library for performing 2D finite element analysis (FEA) of structural systems using the matrix method. It supports:

- **Frame Elements** (beams with bending and axial deformation)
- **Truss Elements** (members with only axial deformation)
- **Spring Elements** (linear springs)
- **Non-linear Analysis** (large deformations and geometric non-linearity)
- **Visualization** (draw structures, supports, loads, and deformed shapes)

## How to Use This Notebook

Scroll through the examples below to see:
1. **Linear Analysis Examples**: Frame, beam, and truss analysis
2. **Non-linear Analysis Examples**: Large displacement analysis
3. **Visualization**: Drawing structures and results

Each example includes:
- Problem description and references
- Code with detailed comments
- Input parameters and geometry
- Printed results (displacements, reactions, member forces)